In [28]:
import pandas as pd

df = pd.read_parquet('5번. 재무비율 및 비재무변수 결측치도 처리.parquet')

## 1. 설립일 (날짜형 데이터)

In [29]:
df["설립일"] = pd.to_datetime(df["설립일"], errors="coerce")

df["업력"] = (
    df["회계년도"]
    - df["설립일"].dt.year
)

print("업력 생성 후 결측치:", df["업력"].isna().sum())

# 산업별 중앙값으로 대체
df["업력"] = (
    df.groupby("통계청 한국표준산업분류 11차(중분류)")["업력"]
      .transform(lambda x: x.fillna(x.median()))
)

# 산업 전체가 결측인 경우 대비
df["업력"] = df["업력"].fillna(df["업력"].median())


print("✅ 설립일 → 업력 변환 완료")
print(f"   업력 NaN : {df['업력'].isnull().sum():,}개")
print(f"   업력 범위: {df['업력'].min():.0f}년 ~ {df['업력'].max():.0f}년")



업력 생성 후 결측치: 207
✅ 설립일 → 업력 변환 완료
   업력 NaN : 0개
   업력 범위: -17년 ~ 127년


In [30]:
df['업력'].describe()

count    308042.000000
mean         16.556512
std          12.501675
min         -17.000000
25%           7.000000
50%          14.000000
75%          23.000000
max         127.000000
Name: 업력, dtype: float64

In [31]:
neg = df[df["업력"] < 0]

print("음수 업력 건수:", len(neg))

print(
    neg[
        ["회사명", "회계년도", "설립일", "업력"]
    ].head(30)
)

음수 업력 건수: 158
                          회사명  회계년도        설립일    업력
1453    윌리스타워스왓슨코리아손해보험중개주식회사  2013 2019-12-29  -6.0
1454    윌리스타워스왓슨코리아손해보험중개주식회사  2014 2019-12-29  -5.0
2666               주식회사솔렉스마케팅  2014 2015-03-15  -1.0
11222             홍성맑은물사랑주식회사  2012 2029-12-22 -17.0
11223             홍성맑은물사랑주식회사  2013 2029-12-22 -16.0
11224             홍성맑은물사랑주식회사  2014 2029-12-22 -15.0
11225             홍성맑은물사랑주식회사  2015 2029-12-22 -14.0
12237              엠큐브홀딩스주식회사  2018 2019-05-21  -1.0
43070                서울이엔지(주)  2019 2020-04-22  -1.0
58206                (주)삼신이엔씨  2014 2015-08-29  -1.0
70239            주식회사컴투게더피알케이  2012 2016-05-10  -4.0
70240            주식회사컴투게더피알케이  2013 2016-05-10  -3.0
70241            주식회사컴투게더피알케이  2014 2016-05-10  -2.0
70242            주식회사컴투게더피알케이  2015 2016-05-10  -1.0
75450              주식회사케이앤에이치  2013 2014-01-25  -1.0
80466               (주)규람타워렌탈  2013 2014-01-16  -1.0
87226                 (주)부천수지  2013 2014-04-25  -1.0
88560              (주)에스앤에스패널  2

In [32]:
df["설립일"].dt.year.describe()

count    307835.000000
mean       2001.871337
std          12.833152
min        1889.000000
25%        1996.000000
50%        2003.000000
75%        2011.000000
max        2029.000000
Name: 설립일, dtype: float64

In [33]:
df["설립일"].dt.year.value_counts().sort_index()

설립일
1889.0       2
1897.0      13
1911.0      13
1916.0       3
1917.0       3
          ... 
2021.0    4649
2022.0    1420
2023.0      25
2024.0      25
2029.0       4
Name: count, Length: 106, dtype: int64

설립일 컬럼 검증 결과, 회계연도보다 미래 시점의 설립일이 다수 존재하여 음수 업력이 발생하였다. 또한 DART 정보와의 불일치 및 결측치가 확인되어 데이터 신뢰성이 낮다고 판단하였으며, 최종 분석에서는 해당 컬럼(업력-설립일)을 제외하였다.

In [34]:
df.drop(columns=["설립일","업력"], inplace=True)

In [35]:
df.shape

(308042, 105)

## 2. 외부감사기관 (범주형 데이터)

In [36]:
# ── 2. 외부감사기관 → 빅4 더미변수 ─────────────────────────

BIG4 = ["삼일", "삼정", "한영", "안진"]

df["빅4감사"] = df["외부감사기관"].apply(
    lambda x: (
        1
        if pd.notna(x)
        and any(b in str(x) for b in BIG4)
        else 0
    )
).astype(int)



print("\n✅ 외부감사기관 → 빅4감사 더미변수 변환 완료")
print(
    df["빅4감사"]
      .value_counts()
      .rename({1: "빅4(1)", 0: "비빅4(0)"})
      .to_string()
)

df.drop(columns=["외부감사기관"], inplace=True)


✅ 외부감사기관 → 빅4감사 더미변수 변환 완료
빅4감사
비빅4(0)    272463
빅4(1)      35579


## 3. 기존 비율에 x 100 하는 단위변환 처리 (산업별 평균 데이터와의 단위 비슷하게 하기 위한 목적)

In [39]:
df.loc[:, '부채비율':].describe()

,부채비율,총부채비율,장기부채비율,장기부채의존도,차입금의존도,순차입금비율,금융부채비율,자기자본비율,유보율,자본잠식률,...,자기자본증가율,부채증가율,영업현금흐름증가율,FCF증가율,ROA변화,영업이익률변화,부채비율변화,유동비율변화,부실라벨_ICR3년,빅4감사
count,308042.000000,308042.000000,308042.000000,308042.000000,308042.000000,308042.000000,308042.000000,308042.000000,3.080420e+05,308042.000000,...,308042.000000,308042.000000,308042.000000,308042.000000,308042.000000,308042.000000,308042.000000,308042.000000,308042.000000,308042.000000
mean,34.403975,0.849067,13.455159,0.225293,0.547156,27.081063,28.404538,0.150937,3.853638e+03,-40.601220,...,0.315918,1.401575,0.531056,-0.990982,-0.006478,0.048297,-1.244732,-3.392700,0.058443,0.115500
std,503.787615,29.311484,329.572102,3.991513,24.793875,477.971777,486.757052,29.311108,9.762593e+04,984.255272,...,38.571315,376.286343,114.411200,162.289700,0.341120,44.461095,316.271090,550.853319,0.234580,0.319625
min,-0.062804,-0.067013,-14.000000,-0.436943,0.000000,-759.000000,0.000000,-8957.000000,-6.463100e+06,-186283.000000,...,-7275.000000,-1.000000,-56712.000000,-56712.000000,-167.164458,-8505.929389,-84412.766917,-218823.713846,0.000000,0.000000
25%,0.623341,0.383990,0.054273,0.022229,0.084425,-0.003860,0.152478,0.171564,7.489661e+01,-24.640000,...,0.000000,-0.076518,-0.133021,-0.045600,-0.021343,-0.025973,-0.199911,-0.180237,0.000000,0.000000
50%,1.693187,0.628693,0.317547,0.105219,0.351991,0.779451,0.947992,0.371308,6.561497e+02,-7.486938,...,0.034970,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,4.828711,0.828439,1.501616,0.289688,0.596984,2.947570,3.179185,0.616010,2.313435e+03,-1.193072,...,0.154481,0.156108,0.227732,0.087982,0.011967,0.018149,0.068000,0.182511,0.000000,0.000000
max,85115.000000,8958.000000,70000.000000,1811.625000,8958.000000,76887.000000,81111.000000,1.067013,1.862830e+07,64631.000000,...,12119.000000,205806.000000,9615.000000,12546.000000,26.572847,8493.584923,34487.600000,91948.147213,1.000000,1.000000


In [40]:
"""
Cell output 14 [DW] 단위 변환 코드
=====================================
- 비율/이익률/증가율 계열: 소수(비율) → % 단위로 × 100
- 회전율 계열: 이미 "회" 단위로 동일 → 변환 없음
- 산업평균과 매핑 불가 컬럼(라벨, 메타): 변환 없음
"""

# ─────────────────────────────────────────────
# 2. × 100 변환이 필요한 컬럼 정의
#    (소수 형태 → % 형태로 통일)
# ─────────────────────────────────────────────

# [자산/자본 비율 계열]
asset_ratio_cols = [
    "부채비율",           # 소수 → %   예) 1.693 → 169.3%
    "총부채비율",         # 소수 → %
    "장기부채비율",       # 소수 → %
    "장기부채의존도",     # 소수 → %
    "차입금의존도",       # 소수 → %   예) 0.352 → 35.2%
    "순차입금비율",       # 소수 → %
    "금융부채비율",       # 소수 → %
    "자기자본비율",       # 소수 → %   예) 0.371 → 37.1%
    "유보율",             # 소수 → %
    "자본잠식률",         # 소수 → %
    "유동비율",           # 소수 → %   예) 1.248 → 124.8%
    "당좌비율_추정",      # 소수 → %   산업평균 '당좌비율'에 대응
    "현금비율",           # 소수 → %   예) 0.132 → 13.2%
    "순운전자본비율",     # 소수 → %
    "비유동비율",         # 소수 → %   예) 1.241 → 124.1%
    "비유동장기적합률",   # 소수 → %   예) 0.852 → 85.2%
    "유형자산비율",       # 소수 → %
    "유형자산부채비율",   # 소수 → %
]

# [수익성 지표 계열]
profitability_cols = [
    "ROA",                # 소수 → %   예) 0.017 → 1.7%
    "ROE",                # 소수 → %   예) 0.104 → 10.4%
    "ROIC",               # 소수 → %
    "총자본영업이익률",   # 소수 → %
    "매출총이익률",       # 소수 → %
    "영업이익률",         # 소수 → %   산업평균 '매출액영업이익률'에 대응
    "순이익률",           # 소수 → %   산업평균 '매출액순이익률'에 대응
    "EBITDA마진",         # 소수 → %   산업평균 'EBITDA대매출액'에 대응
    "영업현금흐름비율",   # 소수 → %
    "현금ROA",            # 소수 → %
    "현금ROE",            # 소수 → %
    "매출원가율",         # 소수 → %   산업평균 '매출원가대매출액'에 대응
    "판관비율",           # 소수 → %
    "감가상각비율",       # 소수 → %   산업평균 '감가상각률'에 대응
    "금융비용부담률",     # 소수 → %   산업평균 '금융비용대매출액'에 대응
    "영업CF_유동부채",    # 소수 → %
    "영업CF_총부채",      # 소수 → %
    "FCF_총자산",         # 소수 → %
]

# [성장성 지표 계열]
growth_cols = [
    "매출액증가율",       # 소수 → %   예) 0.05 → 5%
    "영업이익증가율",     # 소수 → %
    "순이익증가율",       # 소수 → %
    "EBITDA증가율",       # 소수 → %
    "총자산증가율",       # 소수 → %   예) 0.017 → 1.7%
    "유형자산증가율",     # 소수 → %
    "자기자본증가율",     # 소수 → %
    "부채증가율",         # 소수 → %
    "영업현금흐름증가율", # 소수 → %
    "FCF증가율",          # 소수 → %
]

# [변화량 지표 계열]
change_cols = [
    "ROA변화",            # 소수 → %p
    "영업이익률변화",     # 소수 → %p
    "부채비율변화",       # 소수 → %p
    "유동비율변화",       # 소수 → %p
]

# 전체 × 100 대상 컬럼 통합
cols_to_multiply = (
    asset_ratio_cols
    + profitability_cols
    + growth_cols
    + change_cols
)

# ─────────────────────────────────────────────
# 3. 변환 적용 (× 100)
# ─────────────────────────────────────────────
df_converted = df.copy()

for col in cols_to_multiply:
    if col in df_converted.columns:
        df_converted[col] = df_converted[col] * 100
    else:
        print(f"[WARNING] '{col}' 컬럼이 데이터에 없습니다. 건너뜁니다.")

# ─────────────────────────────────────────────
# 4. 변환 안 하는 컬럼 목록 (참고용)
# ─────────────────────────────────────────────
#  회전율 계열: 이미 "회(×)" 단위로 산업평균과 동일
#    - 총자산회전율, 유동자산회전율, 비유동자산회전율
#    - 유형자산회전율, 자기자본회전율, 투하자본회전율
#    - 매출채권회전율, 재고자산회전율, 매입채무회전율
#    - 순운전자본회전율
#  기간(일) 계열: 별도 단위(일)
#    - 매출채권회수기간, 재고자산보유기간, 매입채무지급기간, 현금전환주기_CCC
#  절대금액 계열
#    - FCF
#  이자보상배율: 배수(×) 단위
#  라벨/메타: 부실라벨_ICR3년, 빅4감사

# ─────────────────────────────────────────────
# 5. 변환 결과 검증 (median 기준)
# ─────────────────────────────────────────────
print("=" * 65)
print("변환 결과 검증 (median 행 기준, 산업평균 범위와 비교)")
print("=" * 65)

# CSV는 describe() 결과: row0=count, row1=mean, row2=std, row3=min, row4=25%, row5=median
median_before = df.iloc[5]
median_after  = df_converted.iloc[5]

validation = {
    "부채비율":         ("99~116%",    "자산자본"),
    "자기자본비율":     ("46~52%",     "자산자본"),
    "유동비율":         ("105~162%",   "자산자본"),
    "당좌비율_추정":    ("74~109%",    "자산자본 / '당좌비율'"),
    "현금비율":         ("12~22%",     "자산자본"),
    "비유동비율":       ("92~131%",    "자산자본"),
    "비유동장기적합률": ("70~97%",     "자산자본"),
    "차입금의존도":     ("27~30%",     "자산자본"),
    "ROA":              ("2~8%",       "수익성"),
    "ROE":              ("5~15%",      "수익성"),
    "영업이익률":       ("6~8%",       "손익 / '매출액영업이익률'"),
    "순이익률":         ("3~7%",       "손익 / '매출액순이익률'"),
    "EBITDA마진":       ("10~12%",     "손익 / 'EBITDA대매출액'"),
    "매출원가율":       ("67~68%",     "손익 / '매출원가대매출액'"),
    "감가상각비율":     ("8~10%",      "손익 / '감가상각률'"),
    "금융비용부담률":   ("0.98~1.52%", "손익 / '금융비용대매출액'"),
    "매출액증가율":     ("2~4%",       "성장성"),
    "총자산증가율":     ("2~8%",       "성장성"),
    "유형자산증가율":   ("1~8%",       "성장성"),
    "자기자본증가율":   ("3~9%",       "성장성"),
}

print(f"{'컬럼':<20} {'변환전':>10} {'변환후(×100)':>14}  {'산업평균 범위':<15} {'대응 카테고리'}")
print("-" * 90)
for col, (ref, category) in validation.items():
    before = median_before[col]
    after  = median_after[col]
    print(f"{col:<20} {before:>10.4f} {after:>14.4f}  {ref:<15} {category}")


print(f"   전체 행 수: {len(df_converted)}, 컬럼 수: {len(df_converted.columns)}")
print(f"   × 100 적용 컬럼 수: {len([c for c in cols_to_multiply if c in df_converted.columns])}개")

변환 결과 검증 (median 행 기준, 산업평균 범위와 비교)
컬럼                          변환전      변환후(×100)  산업평균 범위         대응 카테고리
------------------------------------------------------------------------------------------
부채비율                     0.5856        58.5603  99~116%         자산자본
자기자본비율                   0.6307        63.0675  46~52%          자산자본
유동비율                     1.9115       191.1505  105~162%        자산자본
당좌비율_추정                  1.0018       100.1798  74~109%         자산자본 / '당좌비율'
현금비율                     0.1342        13.4236  12~22%          자산자본
비유동비율                    0.7408        74.0804  92~131%         자산자본
비유동장기적합률                 0.6478        64.7754  70~97%          자산자본
차입금의존도                   0.2728        27.2767  27~30%          자산자본
ROA                      0.0206         2.0566  2~8%            수익성
ROE                      0.0325         3.2499  5~15%           수익성
영업이익률                    0.1127        11.2699  6~8%            손익 / '매출액영업이익률'
순이익률                    

In [42]:
df_converted.loc[:, '부채비율':].describe()

,부채비율,총부채비율,장기부채비율,장기부채의존도,차입금의존도,순차입금비율,금융부채비율,자기자본비율,유보율,자본잠식률,...,자기자본증가율,부채증가율,영업현금흐름증가율,FCF증가율,ROA변화,영업이익률변화,부채비율변화,유동비율변화,부실라벨_ICR3년,빅4감사
count,3.080420e+05,308042.000000,3.080420e+05,308042.000000,308042.000000,3.080420e+05,3.080420e+05,308042.000000,3.080420e+05,3.080420e+05,...,3.080420e+05,3.080420e+05,3.080420e+05,3.080420e+05,308042.000000,308042.000000,3.080420e+05,3.080420e+05,308042.000000,308042.000000
mean,3.440398e+03,84.906678,1.345516e+03,22.529263,54.715618,2.708106e+03,2.840454e+03,15.093745,3.853638e+05,-4.060122e+03,...,3.159180e+01,1.401575e+02,5.310563e+01,-9.909817e+01,-0.647778,4.829745,-1.244732e+02,-3.392700e+02,0.058443,0.115500
std,5.037876e+04,2931.148368,3.295721e+04,399.151292,2479.387475,4.779718e+04,4.867571e+04,2931.110777,9.762593e+06,9.842553e+04,...,3.857132e+03,3.762863e+04,1.144112e+04,1.622897e+04,34.112031,4446.109499,3.162711e+04,5.508533e+04,0.234580,0.319625
min,-6.280448e+00,-6.701321,-1.400000e+03,-43.694318,0.000000,-7.590000e+04,0.000000e+00,-895700.000000,-6.463100e+08,-1.862830e+07,...,-7.275000e+05,-1.000000e+02,-5.671200e+06,-5.671200e+06,-16716.445783,-850592.938931,-8.441277e+06,-2.188237e+07,0.000000,0.000000
25%,6.233410e+01,38.398978,5.427294e+00,2.222895,8.442468,-3.860253e-01,1.524775e+01,17.156412,7.489661e+03,-2.464000e+03,...,0.000000e+00,-7.651835e+00,-1.330209e+01,-4.560020e+00,-2.134326,-2.597251,-1.999112e+01,-1.802374e+01,0.000000,0.000000
50%,1.693187e+02,62.869266,3.175472e+01,10.521856,35.199078,7.794506e+01,9.479920e+01,37.130780,6.561497e+04,-7.486938e+02,...,3.496966e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000
75%,4.828711e+02,82.843851,1.501616e+02,28.968777,59.698410,2.947570e+02,3.179185e+02,61.601022,2.313435e+05,-1.193072e+02,...,1.544806e+01,1.561080e+01,2.277315e+01,8.798161e+00,1.196661,1.814896,6.799972e+00,1.825112e+01,0.000000,0.000000
max,8.511500e+06,895800.000000,7.000000e+06,181162.500000,895800.000000,7.688700e+06,8.111100e+06,106.701321,1.862830e+09,6.463100e+06,...,1.211900e+06,2.058060e+07,9.615000e+05,1.254600e+06,2657.284667,849358.492308,3.448760e+06,9.194815e+06,1.000000,1.000000


In [43]:
df = df_converted

## 저장

In [45]:
df.to_parquet('6번. 인코딩 및 데이터 변환.parquet')